# 01. Apollo Baseline

This notebook runs the full cross-validation benchmark on the purified tumor data.
It implements the exact methodology from the Apollo paper: Feature Selection (Top 20) + SVM.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import joblib

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.baselines import ApolloPipeline

In [ ]:
import os
import sys
from pathlib import Path

# Add src to path and fix working directory
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment

# Setup experiment from config
config_name = 'exp_102'
config, output_dir = setup_experiment(config_name)

print(f"✓ Experiment loaded: {config_name}")
print(f"✓ Output directory: {output_dir}")

📋 Experiment ID: default
📝 Model suffix: 


In [ ]:
# --- Load configuration from exp_102.yaml ---
DATA_PATH = config['paths']['processed_file']
SPLIT_FILE = config['paths']['splits_file']
MODEL_DIR = config['paths']['models_baseline_dir']

print(f"Data path: {DATA_PATH}")
print(f"Splits file: {SPLIT_FILE}")
print(f"Model output dir: {MODEL_DIR}")

# Create model directory if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

print("Initializing Loader from config...")
loader = GliomaDataLoader.from_config(
    h5_data_path=DATA_PATH,
    config=config,
    memory_mapped=False
)

print(f"✓ Loader initialized. Total spectra: {loader.X.shape[0]}")   
print(f"✓ Number of folds: {len(loader)}")

NameError: name 'os' is not defined

In [ ]:
import time
import os
import gc
import joblib
import numpy as np
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from src.models.cnn import build_1d_cnn 
from src.models.baselines import ApolloPipeline

# --- Extract model configs ---
apollo_cfg = config['models']['apollo']
rf_cfg = config['models']['random_forest']
xgb_cfg = config['models']['xgboost']
cnn_cfg = config['models']['cnn']

benchmark_results = {
    'Apollo': [],
    'RF_Standard': [],
    'XGBoost': [],
    '1D_CNN': []
}

print("Starting Multi-Model TRTR Benchmark (config-driven)...")
print(f"Models dir: {MODEL_DIR}")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*40}")
    print(f"Processing {fold_name}")
    print(f"{'='*40}")
    
    # Unpack fold data
    X_train = fold_data['train'][0]
    y_train = fold_data['train'][1]
    X_val = fold_data['val'][0]
    y_val = fold_data['val'][1]
    
    # Fix 1-indexed labels if necessary
    if np.min(y_train) == 1:
        y_train = y_train - 1
        y_val = y_val - 1
        
    # --- MODEL 1: Apollo Pipeline ---
    print(f"Training Apollo (n_features={apollo_cfg['n_features']})...")
    apollo_pipeline = ApolloPipeline(
        n_features=apollo_cfg['n_features'],
        random_state=apollo_cfg['random_state']
    )
    apollo_pipeline.fit(X_train, y_train)
    y_pred_apollo = apollo_pipeline.predict(X_val)
    benchmark_results['Apollo'].append(accuracy_score(y_val, y_pred_apollo))
    joblib.dump(apollo_pipeline, f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    del y_pred_apollo
    gc.collect()
    
    # --- MODEL 2: Random Forest ---
    print(f"Training RF (n_est={rf_cfg['n_estimators']})...")
    rf = RandomForestClassifier(
        n_estimators=rf_cfg['n_estimators'],
        class_weight=rf_cfg['class_weight'],
        random_state=rf_cfg['random_state'],
        n_jobs=rf_cfg['n_jobs']
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_val)
    benchmark_results['RF_Standard'].append(accuracy_score(y_val, y_pred_rf))
    joblib.dump(rf, f"{MODEL_DIR}/rf_{fold_name}.joblib")
    del rf, y_pred_rf
    gc.collect()
    
    # --- MODEL 3: XGBoost ---
    print(f"Training XGBoost...")
    xgb_model = XGBClassifier(
        objective=xgb_cfg['objective'],
        num_class=xgb_cfg['num_class'],
        random_state=xgb_cfg['random_state'],
        n_jobs=xgb_cfg['n_jobs'],
        tree_method=xgb_cfg['tree_method']
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_val)
    benchmark_results['XGBoost'].append(accuracy_score(y_val, y_pred_xgb))
    xgb_model.save_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    del xgb_model, y_pred_xgb
    gc.collect()
    
    # --- MODEL 4: 1D-CNN ---
    print(f"Training 1D-CNN (epochs={cnn_cfg['epochs']})...")
    tf.keras.backend.clear_session()
    
    X_train_cnn = X_train.reshape(-1, cnn_cfg['input_shape'][0], cnn_cfg['input_shape'][1])
    X_val_cnn = X_val.reshape(-1, cnn_cfg['input_shape'][0], cnn_cfg['input_shape'][1])
    y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=cnn_cfg['num_classes'])
    y_val_cnn = tf.keras.utils.to_categorical(y_val, num_classes=cnn_cfg['num_classes'])
    
    cnn_model = build_1d_cnn(
        input_shape=tuple(cnn_cfg['input_shape']),
        num_classes=cnn_cfg['num_classes']
    )
    cnn_model.compile(
        optimizer=cnn_cfg['optimizer'],
        loss=cnn_cfg['loss'],
        metrics=['accuracy']
    )
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=cnn_cfg['early_stopping_patience'],
        restore_best_weights=True
    )
    
    cnn_model.fit(X_train_cnn, y_train_cnn,
                  validation_data=(X_val_cnn, y_val_cnn),
                  epochs=cnn_cfg['epochs'],
                  batch_size=cnn_cfg['batch_size'],
                  callbacks=[early_stop],
                  verbose=0)
    
    y_pred_cnn_probs = cnn_model.predict(X_val_cnn, verbose=0)
    y_pred_cnn = np.argmax(y_pred_cnn_probs, axis=1)
    benchmark_results['1D_CNN'].append(accuracy_score(y_val, y_pred_cnn))
    cnn_model.save(f"{MODEL_DIR}/cnn_{fold_name}.keras")
    del cnn_model, X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn, y_pred_cnn_probs, y_pred_cnn
    tf.keras.backend.clear_session()
    gc.collect()

# --- Print Summary ---
print("\n=== TRTR BENCHMARK COMPLETE ===")
for model_name, accuracies in benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f}")

🚀 Starting Memory-Optimized Multi-Model TRTR Benchmark...

🔄 Processing fold_0
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (131932, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_1
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (146045, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_2
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (149039, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_3
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (136360, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...


In [7]:
%reset -f

In [ ]:
import numpy as np
import joblib
import xgboost as xgb
import tensorflow as tf
from sklearn.metrics import accuracy_score
import gc

# Use MODEL_DIR from config (already set in previous cell)

six_class_benchmark_results = {
    'Apollo': [],
    'RF_Standard': [],
    'XGBoost': [],
    '1D_CNN': []
}

binary_benchmark_results = {
    'Apollo (Mapped to Binary)': [],
    'RF_Standard (Mapped to Binary)': [],
    'XGBoost (Mapped to Binary)': [],
    '1D_CNN (Mapped to Binary)': []
}

print("🔍 Testing 6-Class Models on the Binary Task...")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"🔄 Evaluating {fold_name}...")
    
    # Load validation data
    X_val = fold_data['val'][0].astype(np.float32)
    y_val_6_class = fold_data['val'][1]
    
    # Fix 1-indexed to 0-indexed if necessary
    if np.min(y_val_6_class) == 1:
        y_val_6_class = y_val_6_class - 1
        
    # Map True Labels to Binary: 0,1,2 → 0 (Mutant) | 3,4,5 → 1 (Wildtype)
    y_val_binary = np.where(y_val_6_class < 3, 0, 1)
    
    # --- 1. Apollo ---
    apollo = joblib.load(f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    y_pred_apollo_6 = apollo.predict(X_val)
    y_pred_apollo_bin = np.where(y_pred_apollo_6 < 3, 0, 1)
    six_class_benchmark_results['Apollo'].append(accuracy_score(y_val_6_class, y_pred_apollo_6))
    binary_benchmark_results['Apollo (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_apollo_bin))
    
    # --- 2. Random Forest ---
    rf = joblib.load(f"{MODEL_DIR}/rf_{fold_name}.joblib")
    y_pred_rf_6 = rf.predict(X_val)
    y_pred_rf_bin = np.where(y_pred_rf_6 < 3, 0, 1)
    six_class_benchmark_results['RF_Standard'].append(accuracy_score(y_val_6_class, y_pred_rf_6))
    binary_benchmark_results['RF_Standard (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_rf_bin))
    
    # --- 3. XGBoost ---
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    y_pred_xgb_6 = xgb_model.predict(X_val)
    y_pred_xgb_bin = np.where(y_pred_xgb_6 < 3, 0, 1)
    six_class_benchmark_results['XGBoost'].append(accuracy_score(y_val_6_class, y_pred_xgb_6))
    binary_benchmark_results['XGBoost (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_xgb_bin))
    
    # --- 4. 1D-CNN ---
    cnn = tf.keras.models.load_model(f"{MODEL_DIR}/cnn_{fold_name}.keras")
    X_val_cnn = X_val.reshape(-1, 1738, 1)
    y_pred_cnn_probs = cnn.predict(X_val_cnn, verbose=0)
    y_pred_cnn_6 = np.argmax(y_pred_cnn_probs, axis=1)
    y_pred_cnn_bin = np.where(y_pred_cnn_6 < 3, 0, 1)
    six_class_benchmark_results['1D_CNN'].append(accuracy_score(y_val_6_class, y_pred_cnn_6))
    binary_benchmark_results['1D_CNN (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_cnn_bin))
    
    # Cleanup memory
    del apollo, rf, xgb_model, cnn

# Print results
print("\n=== 6-CLASS RESULTS (TRTR) ===")
for model_name, accuracies in six_class_benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f}")

print("\n=== BINARY RESULTS (Mapped from 6-Class Predictions) ===")
for model_name, accuracies in binary_benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>33}: {mean_acc:.4f} ± {std_acc:.4f}")

🔍 Testing 6-Class Models on the Binary Task...


NameError: name 'loader' is not defined